In [1]:
import os
import sys

project_root = os.path.abspath("..")   # if the notebook is inside notebooks/
sys.path.insert(0, project_root)
print(project_root)

c:\Projects\Service Event Framework\Improvement\code


In [2]:
import os
import numpy as np
import pandas as pd
from utils.dbconnection import engine
from utils.config import config
from pathlib import Path

2026-09-21 09:56:40,869 - Logger: Log file: c:\Projects\Service Event Framework\Improvement\code\logs\SEF_Improvement_20260921_095640.log
2026-09-21 09:56:40,888 - SQLAlchemy: Database connection configured successfully!


In [3]:
file_name = 'ServiceEvent_TaskListGroup.sql'
sql_path = os.path.join(config.locations.queries_folder, file_name)
df = pd.read_sql_query(open(sql_path, "r").read(), engine)
df.head()

,DServiceEventID,Serviceeventcode,ServiceEventName,Priority,confidence,TaskListGroup
0,4,GenBearReplace,Generator Bearing Replacement,7.8,10.0,17382
1,4,GenBearReplace,Generator Bearing Replacement,7.8,10.0,19549
2,4,GenBearReplace,Generator Bearing Replacement,7.8,10.0,34869
3,8,PitchCylinderReplace,Pitch Cylinder Replacement,7.6,7.0,17097
4,8,PitchCylinderReplace,Pitch Cylinder Replacement,7.6,7.0,27082


TaskListGroup present in multiple service events

In [4]:
ts_all = df[['Serviceeventcode','TaskListGroup']]
ts_all = ts_all.drop_duplicates()
ts_all

,Serviceeventcode,TaskListGroup
0,GenBearReplace,17382
1,GenBearReplace,19549
2,GenBearReplace,34869
3,PitchCylinderReplace,17097
4,PitchCylinderReplace,27082
...,...,...
1681,HydOilHoseReplace,23873
1682,HydOilHoseReplace,27457
1683,HydOilHoseReplace,49304
1684,SMTErosionRepair,22094


In [5]:
tfg1 = pd.DataFrame(ts_all['TaskListGroup'].value_counts())
tfg1 = tfg1.reset_index()
tfg1.columns = ['TaskListGroup', 'Count']
tfg = tfg1.loc[tfg1['Count'] > 1]
tfg.shape

(45, 2)

In [6]:
tfg.tail()

,TaskListGroup,Count
40,35540,2
41,35544,2
42,35155,2
43,36101,2
44,36102,2


In [7]:
tfg2 = ts_all.loc[ts_all['TaskListGroup'].isin(tfg['TaskListGroup'])]
tfg2.sort_values(by='TaskListGroup', inplace=True)
tfg2.head()

C:\Users\SRVID\AppData\Local\Temp\ipykernel_31596\1693812403.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tfg2.sort_values(by='TaskListGroup', inplace=True)


,Serviceeventcode,TaskListGroup
484,SchedABVisualInspection,11191
703,SchedGMS,11191
710,SchedGMS,11384
138,SchedTrafoInspection,11384
141,SchedTrafoInspection,11730


Add task list group description for better clarity

In [8]:
tasklist_values = tfg2["TaskListGroup"].dropna().astype(str).unique().tolist()

if not tasklist_values:
    raise ValueError("No TaskListGroup values found")

conditions = " OR ".join(
    "SAPGroup LIKE '%" + str(i) + "%'"
    for i in (tasklist_values))

In [9]:
sql = f"""
SELECT distinct
    SAPGroup AS TaskListGroup,
    TaskListDescription AS TaskListDescription
FROM dim.tasklistdetail
WHERE {conditions}
"""

tasklist_detail = pd.read_sql_query(sql, engine)

In [10]:
tasklist_detail['TaskListGroup'] = tasklist_detail['TaskListGroup'].astype(int)

In [11]:
tasklist_detail.head()

,TaskListGroup,TaskListDescription
0,11191,NM54 - 950/900 KW AB -Service Visual GMS
1,11384,V52 850KW E - Service 36 months GMS
2,11730,Transformer inspection V52 EEA
3,12072,Insp. of service lift V80/V90 1.8-2.0 MW
4,12072,Inspection of Avanti service lift


In [12]:
tasklist_detail['TaskListGroup'].nunique()

135

In [13]:
tasklist_detail.shape, tfg2['TaskListGroup'].nunique()

((158, 2), 45)

In [18]:
tasklist_detail = tasklist_detail.groupby(['TaskListGroup']).agg({'TaskListDescription':'first'})
tasklist_detail = tasklist_detail.reset_index()
tasklist_detail.head(10)

,TaskListGroup,TaskListDescription
0,11191,NM54 - 950/900 KW AB -Service Visual GMS
1,11384,V52 850KW E - Service 36 months GMS
2,11730,Transformer inspection V52 EEA
3,12072,Insp. of service lift V80/V90 1.8-2.0 MW
4,12074,Inspection of Avanti service lift
5,12189,V66 1.75/2.0 Internal Crane Inspection
6,12289,Safety Inspection V82/NM82
7,12508,Adjustment of Yaw Controller
8,15708,Blade Inspection with Scope and Camera
9,16626,Transformer Inspection V80-V100 MK7-10


Merge and save the file

Resolve nulls in the final output

In [19]:
tfg2['TaskListGroup'] = tfg2['TaskListGroup'].astype(int)
tfgm = tfg2.merge(tasklist_detail, on='TaskListGroup', how='left')
tfgm

C:\Users\SRVID\AppData\Local\Temp\ipykernel_31596\3777342947.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tfg2['TaskListGroup'] = tfg2['TaskListGroup'].astype(int)


,Serviceeventcode,TaskListGroup,TaskListDescription
0,SchedABVisualInspection,11191,NM54 - 950/900 KW AB -Service Visual GMS
1,SchedGMS,11191,NM54 - 950/900 KW AB -Service Visual GMS
2,SchedGMS,11384,V52 850KW E - Service 36 months GMS
3,SchedTrafoInspection,11384,V52 850KW E - Service 36 months GMS
4,SchedTrafoInspection,11730,Transformer inspection V52 EEA
...,...,...,...
85,TowerAccelerometer_mat,36102,CIM 5036 Inspection of Lightning band
86,BladeWebDisbondRepair,37042,Follow up inspection of TE web scallop
87,BladeInspection,37042,Follow up inspection of TE web scallop
88,BladeWrinkleDelaminatedRepair,43671,Inspect for blade root wrinkles


In [20]:
tfgm.head(20)

,Serviceeventcode,TaskListGroup,TaskListDescription
0,SchedABVisualInspection,11191,NM54 - 950/900 KW AB -Service Visual GMS
1,SchedGMS,11191,NM54 - 950/900 KW AB -Service Visual GMS
2,SchedGMS,11384,V52 850KW E - Service 36 months GMS
3,SchedTrafoInspection,11384,V52 850KW E - Service 36 months GMS
4,SchedTrafoInspection,11730,Transformer inspection V52 EEA
5,SchedGMS,11730,Transformer inspection V52 EEA
6,ServiceLiftInspection,12072,Insp. of service lift V80/V90 1.8-2.0 MW
7,SchedServiceLiftInspection,12072,Insp. of service lift V80/V90 1.8-2.0 MW
8,SchedServiceLiftInspection,12074,Inspection of Avanti service lift
9,ServiceLiftInspection,12074,Inspection of Avanti service lift


In [22]:
tfgm.shape, tfgm['TaskListGroup'].nunique()

((90, 3), 45)

In [24]:
tfgm.to_excel(r"C:\Projects\Service Event Framework\Improvement\code\data\outputs\tasklistgroup_analysis.xlsx", index=False)

PermissionError: [Errno 13] Permission denied: 'C:\\Projects\\Service Event Framework\\Improvement\\code\\data\\outputs\\tasklistgroup_analysis.xlsx'